# Notebook 1 — Data Preparation + MI/mRMR + CPU QAOA

Run this notebook **first**.

- Dataset: `/home/auhspark2/dhiraj/landslide`
- Output: `/home/auhspark2/dhiraj/landslide/Paper1_QAOA_Split`
- This notebook **does not import PyTorch**
- CUDA-Q QAOA uses **`qpp-cpu` only**
- It saves train/validation/holdout splits, normalization, MI, mRMR, exact-QUBO reference, and QAOA-selected feature sets.

When it finishes, **restart the Jupyter kernel before opening Notebook 2**.

In [ ]:
# ============================================================
# CHECK THE FIXED PATH
# ============================================================
from pathlib import Path

DATA_ROOT = Path(r"/home/auhspark2/dhiraj/landslide")
OUTPUT_ROOT = DATA_ROOT / "Paper1_QAOA_Split"

print("DATA_ROOT  :", DATA_ROOT)
print("OUTPUT_ROOT:", OUTPUT_ROOT)

if not DATA_ROOT.exists():
    raise FileNotFoundError(f"Dataset folder not found: {DATA_ROOT}")

## Define all preparation and QAOA functions

    This cell only defines the workflow. It does not start PyTorch or GPU training.

In [ ]:
#!/usr/bin/env python3
"""
Stage 1 — Landslide4Sense data preparation + classical feature selection
          + CPU CUDA-Q QAOA.

This stage intentionally DOES NOT use PyTorch or the NVIDIA GPU.
It exits completely before the deep-learning stage begins.

Dataset root is fixed to:
    /home/auhspark2/dhiraj/landslide
"""

from pathlib import Path
from itertools import combinations
import argparse
import json
import math
import random
import re
import time

import h5py
import numpy as np
import pandas as pd
from sklearn.feature_selection import mutual_info_classif
from sklearn.model_selection import train_test_split

DATA_ROOT_DEFAULT = Path(r"/home/auhspark2/dhiraj/landslide")
CHANNEL_NAMES = [
    "B1","B2","B3","B4","B5","B6","B7","B8",
    "B9","B10","B11","B12","SLOPE","DEM"
]

def extract_id(path: Path) -> int:
    m = re.findall(r"\d+", path.stem)
    return int(m[-1]) if m else -1

def find_named_dir(root: Path, name: str) -> Path:
    direct = root / name
    if direct.exists():
        return direct
    matches = [p for p in root.rglob(name) if p.is_dir()]
    if not matches:
        raise FileNotFoundError(f"Could not find {name} below {root}")
    return sorted(matches, key=lambda p: len(str(p)))[0]

def pair_train(train_dir: Path) -> pd.DataFrame:
    imgs = {extract_id(p): p for p in (train_dir/"img").glob("*.h5")}
    masks = {extract_id(p): p for p in (train_dir/"mask").glob("*.h5")}
    ids = sorted(set(imgs) & set(masks))
    if not ids:
        raise RuntimeError("No paired TrainData image/mask files found.")
    return pd.DataFrame({
        "id": ids,
        "image_path": [str(imgs[i]) for i in ids],
        "mask_path": [str(masks[i]) for i in ids],
    })

def list_images(split_dir: Path) -> pd.DataFrame:
    paths = sorted((split_dir/"img").glob("*.h5"), key=extract_id)
    return pd.DataFrame({
        "id": [extract_id(p) for p in paths],
        "image_path": [str(p) for p in paths],
    })

def mask_stats(records: pd.DataFrame) -> pd.DataFrame:
    rows = []
    for k, row in enumerate(records.itertuples(index=False), 1):
        with h5py.File(row.mask_path, "r") as f:
            y = np.asarray(f["mask"][...], dtype=np.uint8)
        pos = int((y == 1).sum())
        neg = int(y.size - pos)
        rows.append({
            "id": row.id,
            "positive_pixels": pos,
            "negative_pixels": neg,
            "positive_fraction": pos / y.size,
            "contains_landslide": int(pos > 0),
        })
        if k % 250 == 0:
            print(f"  masks checked: {k}/{len(records)}")
    return pd.DataFrame(rows)

def split_labeled(records: pd.DataFrame, seed=42):
    strat = records["contains_landslide"].astype(int)
    tr, tmp = train_test_split(
        records, test_size=0.30, random_state=seed, stratify=strat
    )
    strat2 = tmp["contains_landslide"].astype(int)
    va, te = train_test_split(
        tmp, test_size=0.50, random_state=seed, stratify=strat2
    )
    return tr.reset_index(drop=True), va.reset_index(drop=True), te.reset_index(drop=True)

def compute_mean_std(records: pd.DataFrame):
    sums = np.zeros(14, np.float64)
    sums2 = np.zeros(14, np.float64)
    counts = np.zeros(14, np.int64)
    for k, row in enumerate(records.itertuples(index=False), 1):
        with h5py.File(row.image_path, "r") as f:
            x = np.asarray(f["img"][...], dtype=np.float32).reshape(-1, 14)
        finite = np.isfinite(x)
        clean = np.where(finite, x, 0.0).astype(np.float64)
        sums += clean.sum(axis=0)
        sums2 += (clean * clean).sum(axis=0)
        counts += finite.sum(axis=0)
        if k % 250 == 0:
            print(f"  normalization images: {k}/{len(records)}")
    mean = sums / np.maximum(counts, 1)
    var = sums2 / np.maximum(counts, 1) - mean**2
    std = np.sqrt(np.maximum(var, 1e-12))
    std[std < 1e-6] = 1.0
    return mean.astype(np.float32), std.astype(np.float32)

def balanced_pixel_sample(records, mean, limit=200_000, per_patch=256, seed=42):
    rng = np.random.default_rng(seed)
    pos_chunks, neg_chunks = [], []
    for k, row in enumerate(records.itertuples(index=False), 1):
        with h5py.File(row.image_path, "r") as f:
            x = np.asarray(f["img"][...], dtype=np.float32).reshape(-1, 14)
        with h5py.File(row.mask_path, "r") as f:
            y = np.asarray(f["mask"][...], dtype=np.uint8).reshape(-1)

        p = np.flatnonzero(y == 1)
        n = np.flatnonzero(y == 0)
        if len(p):
            sel = rng.choice(p, min(per_patch, len(p)), replace=False)
            pos_chunks.append(x[sel])
        if len(n):
            sel = rng.choice(n, min(per_patch, len(n)), replace=False)
            neg_chunks.append(x[sel])

        if k % 250 == 0:
            print(f"  feature-sampling images: {k}/{len(records)}")

    pos = np.concatenate(pos_chunks) if pos_chunks else np.empty((0,14), np.float32)
    neg = np.concatenate(neg_chunks) if neg_chunks else np.empty((0,14), np.float32)
    each = min(len(pos), len(neg), limit // 2)
    pos = pos[rng.choice(len(pos), each, replace=False)]
    neg = neg[rng.choice(len(neg), each, replace=False)]
    X = np.concatenate([pos, neg])
    y = np.concatenate([np.ones(each, np.uint8), np.zeros(each, np.uint8)])
    perm = rng.permutation(len(y))
    X, y = X[perm], y[perm]

    for j in range(14):
        bad = ~np.isfinite(X[:, j])
        X[bad, j] = mean[j]
    return X.astype(np.float32), y

def mrmr_select(relevance, redundancy, k):
    selected = [int(np.argmax(relevance))]
    remaining = [i for i in range(len(relevance)) if i not in selected]
    while len(selected) < k:
        score = {}
        for j in remaining:
            red = np.mean([redundancy[j, s] for s in selected])
            score[j] = float(relevance[j] - red)
        best = max(score, key=score.get)
        selected.append(best)
        remaining.remove(best)
    return selected

def build_qubo(relevance, redundancy, k, alpha=1.0, beta=0.35, penalty=2.5):
    linear = -alpha * relevance + penalty * (1 - 2*k)
    pair = np.zeros((len(relevance), len(relevance)), dtype=float)
    for i in range(len(relevance)):
        for j in range(i+1, len(relevance)):
            pair[i,j] = beta * redundancy[i,j] + 2*penalty
    return linear.astype(float), pair

def qubo_cost(bits, linear, pair):
    x = np.asarray(bits, dtype=float)
    c = float(linear @ x)
    for i in range(len(x)):
        for j in range(i+1, len(x)):
            c += float(pair[i,j] * x[i] * x[j])
    return c

def exact_qubo(linear, pair, k):
    best_subset, best_cost = None, np.inf
    for combo in combinations(range(len(linear)), k):
        bits = np.zeros(len(linear), dtype=np.uint8)
        bits[list(combo)] = 1
        c = qubo_cost(bits, linear, pair)
        if c < best_cost:
            best_subset, best_cost = list(combo), c
    return best_subset, float(best_cost)

def qubo_to_ising(linear, pair):
    h = -0.5 * linear.copy()
    J = np.zeros_like(pair)
    for i in range(len(linear)):
        for j in range(i+1, len(linear)):
            q = pair[i,j]
            J[i,j] = 0.25*q
            h[i] -= 0.25*q
            h[j] -= 0.25*q
    return h, J

def run_qaoa_cpu(relevance, redundancy, k, p=2, shots=4096, max_iter=120, seed=42):
    """
    Run QAOA on CUDA-Q qpp-cpu intentionally.
    This avoids opening an NVIDIA GPU context before PyTorch training.
    """
    import cudaq
    from cudaq import spin

    cudaq.set_target("qpp-cpu")
    cudaq.set_random_seed(seed)

    linear, pair = build_qubo(relevance, redundancy, k)
    h, J = qubo_to_ising(linear, pair)
    n = len(linear)

    src, dst, ew = [], [], []
    for i in range(n):
        for j in range(i+1, n):
            if abs(J[i,j]) > 1e-12:
                src.append(i); dst.append(j); ew.append(float(J[i,j]))
    hw = [float(v) for v in h]

    @cudaq.kernel
    def zz(q0: cudaq.qubit, q1: cudaq.qubit, angle: float):
        x.ctrl(q0, q1)
        rz(2.0 * angle, q1)
        x.ctrl(q0, q1)

    @cudaq.kernel
    def circuit(
        qubit_count: int,
        layers: int,
        hweights: list[float],
        esrc: list[int],
        edst: list[int],
        eweights: list[float],
        theta: list[float],
    ):
        q = cudaq.qvector(qubit_count)
        h(q)
        for layer in range(layers):
            gamma = theta[layer]
            beta = theta[layer + layers]
            for i in range(qubit_count):
                rz(2.0 * gamma * hweights[i], q[i])
            for e in range(len(esrc)):
                zz(q[esrc[e]], q[edst[e]], gamma * eweights[e])
            for i in range(qubit_count):
                rx(2.0 * beta, q[i])

    H = 0
    for i, coeff in enumerate(h):
        if abs(coeff) > 1e-12:
            H += float(coeff) * spin.z(i)
    for i in range(n):
        for j in range(i+1, n):
            if abs(J[i,j]) > 1e-12:
                H += float(J[i,j]) * spin.z(i) * spin.z(j)

    rng = np.random.default_rng(seed)
    opt = cudaq.optimizers.NelderMead()
    opt.max_iterations = max_iter
    opt.initial_parameters = rng.uniform(-np.pi/8, np.pi/8, 2*p).tolist()

    history = []
    def objective(theta):
        value = cudaq.observe(
            circuit, H, n, p, hw, src, dst, ew, theta
        ).expectation()
        history.append(float(value))
        return float(value)

    start = time.perf_counter()
    expectation, theta = opt.optimize(dimensions=2*p, function=objective)
    counts = cudaq.sample(
        circuit, n, p, hw, src, dst, ew, theta, shots_count=shots
    )
    runtime = time.perf_counter() - start

    candidates = []
    for key in counts:
        raw = str(key).replace(" ","").zfill(n)
        if len(raw) != n or set(raw)-{"0","1"}:
            continue
        freq = int(counts[key])
        for ss in (raw, raw[::-1]):
            bits = np.array([int(ch) for ch in ss], np.uint8)
            if int(bits.sum()) == k:
                candidates.append((
                    qubo_cost(bits, linear, pair),
                    -freq,
                    np.flatnonzero(bits).tolist(),
                    ss,
                    freq
                ))
    if not candidates:
        raise RuntimeError(f"No feasible k={k} state sampled. Increase shots.")
    candidates.sort(key=lambda z:(z[0], z[1]))
    best = candidates[0]
    return {
        "k": k, "p": p,
        "subset": best[2],
        "selected_channels": [CHANNEL_NAMES[i] for i in best[2]],
        "qubo_cost": float(best[0]),
        "bitstring": best[3],
        "sample_count": int(best[4]),
        "expectation": float(expectation),
        "runtime_seconds": float(runtime),
        "history": history,
    }

def main():
    # JUPYTER VERSION: no terminal arguments are required.
    data_root = Path(r"/home/auhspark2/dhiraj/landslide")
    out = data_root / "Paper1_QAOA_Split"
    (out/"01_dataset").mkdir(parents=True, exist_ok=True)
    (out/"02_feature_selection").mkdir(parents=True, exist_ok=True)
    (out/"03_models").mkdir(parents=True, exist_ok=True)
    (out/"04_metrics").mkdir(parents=True, exist_ok=True)
    (out/"05_figures").mkdir(parents=True, exist_ok=True)
    (out/"06_predictions").mkdir(parents=True, exist_ok=True)

    print("="*80)
    print("STAGE 1 — DATA PREPARATION + CPU QAOA")
    print("DATA_ROOT:", data_root)
    print("OUTPUT   :", out)
    print("="*80)

    train_dir = find_named_dir(data_root, "TrainData")
    valid_dir = find_named_dir(data_root, "ValidData")
    test_dir = find_named_dir(data_root, "TestData")

    records = pair_train(train_dir)
    valid_rec = list_images(valid_dir)
    test_rec = list_images(test_dir)

    print("Labeled training patches:", len(records))
    print("Official validation     :", len(valid_rec))
    print("Official test           :", len(test_rec))

    stats_path = out/"01_dataset"/"mask_statistics.csv"
    if stats_path.exists():
        stats = pd.read_csv(stats_path)
    else:
        stats = mask_stats(records)
        stats.to_csv(stats_path, index=False)
    records = records.merge(stats, on="id", how="left")

    tr, va, te = split_labeled(records, 42)
    tr.to_csv(out/"01_dataset"/"train_records.csv", index=False)
    va.to_csv(out/"01_dataset"/"validation_records.csv", index=False)
    te.to_csv(out/"01_dataset"/"holdout_records.csv", index=False)
    valid_rec.to_csv(out/"01_dataset"/"official_valid_records.csv", index=False)
    test_rec.to_csv(out/"01_dataset"/"official_test_records.csv", index=False)

    norm_path = out/"01_dataset"/"normalization.json"
    if norm_path.exists():
        norm = json.loads(norm_path.read_text())
        mean = np.array(norm["mean"], np.float32)
        std = np.array(norm["std"], np.float32)
    else:
        print("Computing training-only normalization...")
        mean, std = compute_mean_std(tr)
        norm_path.write_text(json.dumps({
            "channels": CHANNEL_NAMES,
            "mean": mean.tolist(), "std": std.tolist()
        }, indent=2))

    sample_path = out/"02_feature_selection"/"balanced_pixels.npz"
    if sample_path.exists():
        d = np.load(sample_path)
        X, y = d["X"], d["y"]
    else:
        print("Building balanced pixel sample...")
        X, y = balanced_pixel_sample(tr, mean)
        np.savez_compressed(sample_path, X=X, y=y)

    Xz = (X - mean[None,:]) / std[None,:]
    mi = mutual_info_classif(Xz, y, random_state=42)
    mi_norm = mi / (mi.max()+1e-12)
    corr = np.abs(np.corrcoef(Xz, rowvar=False))
    corr = np.nan_to_num(corr)
    np.fill_diagonal(corr, 0.0)

    pd.DataFrame({
        "index": range(14),
        "channel": CHANNEL_NAMES,
        "MI": mi,
        "MI_normalized": mi_norm
    }).to_csv(out/"02_feature_selection"/"MI_scores.csv", index=False)
    pd.DataFrame(corr, index=CHANNEL_NAMES, columns=CHANNEL_NAMES).to_csv(
        out/"02_feature_selection"/"correlation_matrix.csv"
    )

    MI8 = np.argsort(mi_norm)[::-1][:8].tolist()
    MRMR8 = mrmr_select(mi_norm, corr, 8)

    exact_info = {}
    for k in [6,8,10]:
        linear, pair = build_qubo(mi_norm, corr, k)
        subset, cost = exact_qubo(linear, pair, k)
        exact_info[str(k)] = {
            "subset": subset,
            "channels": [CHANNEL_NAMES[i] for i in subset],
            "cost": cost
        }

    qaoa_info = {}
    qrows = []
    # Run p=1,2,3 for each k; choose lowest sampled QUBO cost.
    for k in [6,8,10]:
        runs = []
        for p in [1,2,3]:
            print(f"QAOA CPU: k={k}, p={p}")
            r = run_qaoa_cpu(
                mi_norm, corr, k=k, p=p,
                shots=4096, max_iter=120,
                seed=42 + 10*k + p
            )
            r["exact_cost"] = exact_info[str(k)]["cost"]
            r["gap"] = (
                (r["qubo_cost"] - r["exact_cost"]) /
                (abs(r["exact_cost"])+1e-12)
            )
            runs.append(r)
            qrows.append({
                "k": k, "p": p,
                "qubo_cost": r["qubo_cost"],
                "exact_cost": r["exact_cost"],
                "normalized_gap": r["gap"],
                "runtime_seconds": r["runtime_seconds"],
                "selected_channels": ", ".join(r["selected_channels"])
            })
            pd.DataFrame({
                "evaluation": range(len(r["history"])),
                "expectation": r["history"]
            }).to_csv(
                out/"02_feature_selection"/f"qaoa_history_k{k}_p{p}.csv",
                index=False
            )
        best = min(runs, key=lambda z:z["qubo_cost"])
        qaoa_info[str(k)] = best

    pd.DataFrame(qrows).to_csv(
        out/"02_feature_selection"/"QAOA_summary.csv", index=False
    )

    feature_sets = {
        "ALL_14": list(range(14)),
        "MI_8": MI8,
        "mRMR_8": MRMR8,
        "QAOA_6": qaoa_info["6"]["subset"],
        "QAOA_8": qaoa_info["8"]["subset"],
        "QAOA_10": qaoa_info["10"]["subset"],
    }

    payload = {
        "channel_names": CHANNEL_NAMES,
        "feature_sets": feature_sets,
        "exact_qubo": exact_info,
        "qaoa": {
            k: {
                key: value for key, value in v.items()
                if key != "history"
            } for k,v in qaoa_info.items()
        }
    }
    (out/"02_feature_selection"/"feature_sets.json").write_text(
        json.dumps(payload, indent=2)
    )

    print("\nFINAL FEATURE SETS")
    for name, subset in feature_sets.items():
        print(f"{name:10s}:", [CHANNEL_NAMES[i] for i in subset])

    print("\nStage 1 completed successfully.")
    print("The process will now exit, releasing ALL CUDA-Q memory.")



## Run Notebook 1

    This is the only execution call needed for the complete preparation/QAOA stage.

In [ ]:
# Run complete data preparation + CPU QAOA.
main()

print("\n" + "=" * 80)
print("NOTEBOOK 1 FINISHED")
print("=" * 80)
print("Saved results under:")
print(r"/home/auhspark2/dhiraj/landslide/Paper1_QAOA_Split")
print("\nNOW RESTART THE JUPYTER KERNEL before opening Notebook 2.")